In [1]:
import os
import pandas as pd

# Directory containing subdirectories of experiments
main_dir = "Runs_09-23"
column_to_summarize = "no_filter"

In [2]:
# Function to extract threshold value from directory name
def get_threshold_from_dir(dir_name):
    return dir_name.split("_")[-1]

# Function to process a run directory and collect data for all thresholds
def process_run(run_path):
    threshold_dirs = [d for d in os.listdir(run_path) if os.path.isdir(os.path.join(run_path, d)) and d.startswith("threshold_")]
    run_data = {}
    
    for threshold_dir in threshold_dirs:
        threshold_value = get_threshold_from_dir(threshold_dir)
        csv_file = f"final_stats_threshold_{threshold_value}.csv"
        csv_path = os.path.join(run_path, threshold_dir, csv_file)
        
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path, index_col=0)
            if column_to_summarize in df.columns:
                # Extract required metrics
                run_data[threshold_value] = df[column_to_summarize]
    
    return run_data

# Main function to collect data from all experiments and runs
def collect_data(main_dir):
    # Dictionary to hold the data, outer dict for subdirectory, inner dict for threshold data
    all_data = {}

    sub_dirs = [d for d in os.listdir(main_dir) if os.path.isdir(os.path.join(main_dir, d))]
    for sub_dir in sub_dirs:
        sub_dir_path = os.path.join(main_dir, sub_dir)
        run_dirs = [d for d in os.listdir(sub_dir_path) if os.path.isdir(os.path.join(sub_dir_path, d))]
        for run_dir in run_dirs:
            run_path = os.path.join(sub_dir_path, run_dir)
            run_data = process_run(run_path)
            
            if run_data:
                # Store the data by subdirectory, run, and threshold
                if sub_dir not in all_data:
                    all_data[sub_dir] = {}
                all_data[sub_dir][run_dir] = run_data

    return all_data

# Convert collected data into a DataFrame with multi-level columns
def create_dataframe_and_save(all_data):
    dataframe_dict = {}
    for sub_dir, run_data in all_data.items():
        # Initialize an empty DataFrame to store the final combined results
        df_combined = pd.DataFrame()
        threshold_list = []
        counter = 1
        for run, thresholds in run_data.items():
            for threshold, metrics in thresholds.items():
                # Create multi-level column with format (threshold, run)
                column_name = (threshold, f"run_{counter}")
                df_combined[column_name] = metrics
                threshold_list.append(str(threshold))
            counter += 1

        # # For each experiment (sub_dir), calculate average across all runs for each threshold
        for threshold in set(threshold_list):
            threshold_cols = [col for col in df_combined.columns if threshold in col]
            avg_column_name = (threshold, f"run_average")
            df_combined[avg_column_name] = df_combined[threshold_cols].mean(axis=1)

        # Sort the columns for better readability
        df_combined = df_combined.sort_index(axis=1, level=[0, 1])
        save_to_csv(df_combined, os.path.join(main_dir, f"{sub_dir}.csv"))
        dataframe_dict[sub_dir] = df_combined
    return dataframe_dict

# Save the DataFrame to a CSV
def save_to_csv(df, output_file="aggregated_results.csv"):
    df.to_csv(output_file)


In [3]:
data = collect_data(main_dir)
df_dict = create_dataframe_and_save(data)
# save_to_csv(df_result)
# print("Data aggregation completed and saved to 'aggregated_results.csv'")

In [ ]:
df_dict.keys()